# 04 Gold transformations
1. Load silver incidents from object storage
2. Apply training quality filters (historical state, dedupe, required fields)
3. Build model features and labels (including class ids)
4. Create deterministic train/validation/test splits
5. Write lean artifacts: train.parquet, valid.parquet, test.parquet, label_mapping.json
6. Write dataset_card.md and print final evidence metrics

In [46]:
# Imports
from pathlib import Path
import io
import json
from datetime import datetime, timezone

import pandas as pd
from minio import Minio
from sklearn.model_selection import train_test_split

In [47]:
# Read .env for credentials
def load_env_file(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing env file: {path.resolve()}")

    env = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        env[k.strip()] = v.strip()
    return env

# Env file location in repo
ENV_FILE = Path("../docker/.env")
# Storage endpoint and config
MINIO_ENDPOINT = "localhost:9000"
MINIO_SECURE = False

# Files locations in buckets
SILVER_BUCKET = "incident-pipeline-test"
SILVER_PREFIX = "silver/incidents"
SILVER_OBJECT = f"{SILVER_PREFIX}/incidents.parquet"
PROFILE_BUCKET = "data-profile-test"
PROFILE_PREFIX = "silver/incidents"
PROFILE_SUMMARY_OBJECT = f"{PROFILE_PREFIX}/silver_dq_summary.json"
GOLD_BUCKET = "incident-pipeline-test"
GOLD_PREFIX = "gold/incidents"
GOLD_OBJECT = f"{GOLD_PREFIX}/incidents.parquet"

# Authenticate to file storage list buckets to confirm connection
env = load_env_file(ENV_FILE)
client = Minio(
    MINIO_ENDPOINT,
    access_key=env["MINIO_ROOT_USER"],
    secret_key=env["MINIO_ROOT_PASSWORD"],
    secure=MINIO_SECURE,
)

print("Connected buckets:", [b.name for b in client.list_buckets()])

Connected buckets: ['data-profile-test', 'incident-pipeline', 'incident-pipeline-test', 'mlflow-artifacts']


# 1) Load silver data from storage


In [48]:
# Load silver parquet from storage
resp = client.get_object(SILVER_BUCKET, SILVER_OBJECT)
try:
    silver_df = pd.read_parquet(io.BytesIO(resp.read()))
finally:
    resp.close()
    resp.release_conn()

# Print number of rows and sample of the data
print(f"Rows: {len(silver_df)}")
print(f"Columns: {silver_df.columns.tolist()}")
silver_df.head(3)

Rows: 10000
Columns: ['sys_id', 'number', 'task_effective_number', 'sys_class_name', 'state', 'incident_state', 'priority', 'impact', 'urgency', 'severity', 'approval', 'escalation', 'notify', 'opened_at', 'resolved_at', 'closed_at', 'activity_due', 'due_date', 'sys_created_on', 'sys_updated_on', 'sys_created_by', 'sys_updated_by', 'opened_by', 'caller_id', 'resolved_by', 'closed_by', 'assignment_group', 'assigned_to', 'business_service', 'cmdb_ci', 'category', 'subcategory', 'contact_type', 'short_description', 'description', 'comments', 'work_notes', 'comments_and_work_notes', 'work_notes_list', 'parent_incident', 'problem_id', 'rfc', 'correlation_id', 'correlation_display', 'active', 'knowledge', 'made_sla', 'close_code', 'close_notes', 'calendar_duration', 'business_duration', 'calendar_stc', 'business_stc', 'sys_mod_count', 'reassignment_count', 'reopen_count', 'sla_due', 'sys_domain', 'sys_domain_path', 'watch_list', 'group_list', 'additional_assignee_list', 'u_system', 'u_system

,sys_id,number,task_effective_number,sys_class_name,state,incident_state,priority,impact,urgency,severity,...,u_system,u_system_criticality,u_initial_assignment_group,u_suggested_assignment_group,u_suggested_category,u_suggested_subcategory,u_outage_day,u_outage_system,bronze_run_id,ingested_at_utc
0,ea5947653a4822a6b11248e0d580bb5e,INC1205252,INC1205252,Incident,Closed,Closed,2 - High,1 - High,2 - High,3 - Low,...,MFA,High,Identity and User Access,Identity and User Access,Access,Account locked,false,,20260308T205812Z,2026-03-08 20:58:17.858483+00:00
1,e34a1ba1cef2676aa6b963f3a78f59d6,INC1202323,INC1202323,Incident,Closed,Closed,2 - High,1 - High,2 - High,2 - High,...,SIEM Alerts,High,Security Operations,Security Operations,Security,Malware alert,false,,20260308T205812Z,2026-03-08 20:58:17.858483+00:00
2,a6d47a1b61ca531e0d0a013431e6fd3e,INC1206330,INC1206330,Incident,Closed,Closed,4 - Low,3 - Low,3 - Low,3 - Low,...,Microsoft Fabric Pipelines,Medium,App Support - Microsoft Fabric,App Support - Microsoft Fabric,Data,Pipeline failure,false,,20260308T205812Z,2026-03-08 20:58:17.858483+00:00


In [49]:
# Define required columns and check they are present in the data
CORE_REQUIRED = ["sys_id", "sys_updated_on", "state", "assignment_group"]
TEXT_COLS = ["short_description", "description"]

# Required columns must exist and both text columns must exist
REQUIRED_PRESENT = CORE_REQUIRED + TEXT_COLS
missing = [c for c in REQUIRED_PRESENT if c not in silver_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")
print("All required columns found")

All required columns found


# 2) Apply training quality filters
- verify historical state
- remove duplicates
- check for required fields


In [50]:
# Count open/closed and active true/false, then keep only closed/completed incidents
state_norm = silver_df["state"].astype(str).str.strip().str.lower()
active_norm = silver_df["active"].astype(str).str.strip().str.lower()

open_count = (state_norm == "open").sum()
closed_count = (state_norm == "closed").sum()
active_true_count = active_norm.isin(["true", "1", "yes", "y"]).sum()
active_false_count = active_norm.isin(["false", "0", "no", "n"]).sum()

print(f"Open records: {open_count}")
print(f"Closed records: {closed_count}")
print(f"Active=True records: {active_true_count}")
print(f"Active=False records: {active_false_count}")

keep_mask = state_norm.isin(["closed", "completed"])
silver_filtered_df = silver_df.loc[keep_mask].copy()

print(f"Filtered final record count: {len(silver_filtered_df)}")

Open records: 0
Closed records: 10000
Active=True records: 0
Active=False records: 10000
Filtered final record count: 10000


In [51]:
# Remove duplicate incidents by sys_id
duplicate_mask = silver_filtered_df.duplicated(subset=["sys_id"], keep="first")
duplicate_count = duplicate_mask.sum()
silver_filtered_df = silver_filtered_df.loc[~duplicate_mask].copy()

print(f"Duplicate sys_id rows removed: {duplicate_count}")
print(f"Record count after de-duplication: {len(silver_filtered_df)}")

Duplicate sys_id rows removed: 0
Record count after de-duplication: 10000


In [52]:
# Remove rows with null/blank values in required training columns
required_view = silver_filtered_df.copy()

# core missing/blank
missing_core = required_view[CORE_REQUIRED].isna().any(axis=1)
for c in CORE_REQUIRED:
    missing_core = missing_core | required_view[c].astype(str).str.strip().eq("")

# Drop a row if both text fields are blank, otherwise keep
short_blank = required_view["short_description"].fillna("").astype(str).str.strip().eq("")
desc_blank  = required_view["description"].fillna("").astype(str).str.strip().eq("")
missing_text = short_blank & desc_blank

drop_mask = missing_core | missing_text
removed = int(drop_mask.sum())

silver_filtered_df = silver_filtered_df.loc[~drop_mask].copy()

print(f"Rows removed (missing core OR both texts blank): {removed}")
print(f"Record count after cleanup: {len(silver_filtered_df)}")

Rows removed (missing core OR both texts blank): 0
Record count after cleanup: 10000


In [53]:
# Create a dated folder so each training export is reproducible
dataset_version = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
GOLD_TRAINING_PREFIX = f"gold/training/{dataset_version}"

print("Gold training prefix:", f"s3://{GOLD_BUCKET}/{GOLD_TRAINING_PREFIX}")

Gold training prefix: s3://incident-pipeline-test/gold/training/20260308_205827


# 3) Build model features and labels


In [54]:
# Work on a copy so we keep the filtered source frame as is for reference
df = silver_filtered_df.copy()

# Build the final text feature from both the short and long ticket content
df["text"] = (
    df["short_description"].fillna("").astype(str).str.strip()
    + "\n" +
    df["description"].fillna("").astype(str).str.strip()
).str.strip()

# Check text completeness and basic length distribution
print("Empty combined text rate:", f"{df['text'].eq('').mean():.2%}")
df["text_len"] = df["text"].str.len()
df["text_len"].describe(percentiles=[0.5, 0.75, 0.9, 0.95])

Empty combined text rate: 0.00%


count    10000.000000
mean       610.598500
std        123.311502
min        353.000000
50%        617.000000
75%        686.000000
90%        769.000000
95%        826.000000
max       1006.000000
Name: text_len, dtype: float64

In [55]:
# Fold very small classes into a manual review bucket
MANUAL_REVIEW_GROUP = "manual_review_group"
MIN_CLASS_COUNT = 20 # min number of records per class

# Clean source labels before frequency checks
df["label_clean"] = df["assignment_group"].astype(str).str.strip()

# Identify low frequency classes that may break splits
counts = df["label_clean"].value_counts()
rare = counts[counts < MIN_CLASS_COUNT]

# Print class distribution and rare classes
print("Unique classes (clean):", df["label_clean"].nunique())
print(f"Rare classes (< {MIN_CLASS_COUNT}):", len(rare))

if len(rare) > 0:
    display(rare.to_frame("count"))

# Build final label column used for training
df["label_final"] = df["label_clean"].where(~df["label_clean"].isin(rare.index), MANUAL_REVIEW_GROUP)
print("Unique classes (final):", df["label_final"].nunique())

Unique classes (clean): 12
Rare classes (< 20): 0
Unique classes (final): 12


In [56]:
# Create mappings to convert labels to integer ids for modelling and splits
classes = sorted(df["label_final"].unique().tolist())
label_to_id = {c: i for i, c in enumerate(classes)}
id_to_label = {i: c for c, i in label_to_id.items()}

# Attach integer class id for modelling and split
df["label_id"] = df["label_final"].map(label_to_id).astype(int)

print("Class count:", len(classes))
print("First 10 classes:", classes[:10])

Class count: 12
First 10 classes: ['App Support - ERP', 'App Support - Finance', 'App Support - HRIS', 'App Support - M365', 'App Support - Microsoft Fabric', 'App Support - Power BI', 'App Support - Power Platform', 'End User Compute', 'Identity and User Access', 'Integration & Middleware']


# 4) Create train/valid/test splits


In [57]:
# Random seed for reproducible splits
RANDOM_STATE = 42

# Hold out 10% test set on label_id to preserve class distribution
train_val, test_df = train_test_split(
    df, test_size=0.10, random_state=RANDOM_STATE, stratify=df["label_id"]
)

# 10% for validation from the remaining 90%
valid_rel = 0.10 / 0.90
train_df, valid_df = train_test_split(
    train_val, test_size=valid_rel, random_state=RANDOM_STATE, stratify=train_val["label_id"]
)

# Final splits 90/10/10
print("Train/Valid/Test:", len(train_df), len(valid_df), len(test_df))

# Check for leakage by unique sys_id field across each split
overlap = (
    (set(train_df["sys_id"]) & set(valid_df["sys_id"])) |
    (set(train_df["sys_id"]) & set(test_df["sys_id"])) |
    (set(valid_df["sys_id"]) & set(test_df["sys_id"]))
)
print("Any sys_id overlap?:", bool(overlap))

Train/Valid/Test: 7999 1001 1000
Any sys_id overlap?: False


In [58]:
# Validate train, test, and validation splits
cols = ["sys_id", "sys_updated_on", "text", "label_final", "label_id"]

print("Train preview:")
display(train_df[cols].head(2))
print("Validation preview:")
display(valid_df[cols].head(2))
print("Test preview:")
display(test_df[cols].head(2))



Train preview:


,sys_id,sys_updated_on,text,label_final,label_id
6949,b602c8005114e06ad4bd4ed9101a7445,2025-05-21 11:28:54+00:00,This is broken and slowing us down - really fr...,App Support - Power Platform,6
3101,c4099ec67af612fc97b2e35ce2806f0d,2024-08-14 19:44:32+00:00,Access/connection issue again\nThis is really ...,App Support - Power Platform,6


Validation preview:


,sys_id,sys_updated_on,text,label_final,label_id
4931,e63e5d01fcd3e483a551eefd33330790,2024-12-30 18:29:23+00:00,Keeps failing and I’m stuck - really frustrati...,App Support - Power BI,5
2759,505f7b4bb67454e76509bf7b747dd2b1,2024-07-17 13:14:55+00:00,Blocked on a basic task - really frustrating\n...,Identity and User Access,8


Test preview:


,sys_id,sys_updated_on,text,label_final,label_id
9973,31bc4baab97e884968db5bb6b31cf9fe,2025-12-31 13:35:01+00:00,Blocked on a basic task - really frustrating\n...,App Support - Power Platform,6
4595,ebd82eb8327e9e415caebfa80dd6c1ad,2024-12-03 14:57:22+00:00,Going in circles with an error - really frustr...,App Support - M365,3


# 5) Write lean gold artifacts to storage


In [59]:
# Save minimal modeling columns for each split
if not client.bucket_exists(GOLD_BUCKET):
    client.make_bucket(GOLD_BUCKET)
    print(f"Created bucket: {GOLD_BUCKET}")
else:
    print(f"Using existing bucket: {GOLD_BUCKET}")

artifact_objects = {}

for name, split_df in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    buf = io.BytesIO()
    split_df[cols].to_parquet(buf, index=False)
    split_bytes = buf.getvalue()

    object_name = f"{GOLD_TRAINING_PREFIX}/{name}.parquet"
    client.put_object(
        bucket_name=GOLD_BUCKET,
        object_name=object_name,
        data=io.BytesIO(split_bytes),
        length=len(split_bytes),
        content_type="application/octet-stream",
    )
    artifact_objects[name] = object_name
    print(f"Wrote: s3://{GOLD_BUCKET}/{object_name}")

# Output label mapping so training/inference use the same class ids
label_mapping = {
    "dataset_version": dataset_version,
    "label_field_source": "assignment_group",
    "label_field_final": "label_final",
    "manual_review_group": MANUAL_REVIEW_GROUP,
    "min_class_count": MIN_CLASS_COUNT,
    "classes": classes,
    "label_to_id": label_to_id,
    "id_to_label": id_to_label,
}
label_mapping_bytes = json.dumps(label_mapping, indent=2).encode("utf-8")
label_mapping_object = f"{GOLD_TRAINING_PREFIX}/label_mapping.json"
client.put_object(
    bucket_name=GOLD_BUCKET,
    object_name=label_mapping_object,
    data=io.BytesIO(label_mapping_bytes),
    length=len(label_mapping_bytes),
    content_type="application/json",
)
artifact_objects["label_mapping"] = label_mapping_object
print(f"Wrote: s3://{GOLD_BUCKET}/{label_mapping_object}")



Using existing bucket: incident-pipeline-test
Wrote: s3://incident-pipeline-test/gold/training/20260308_205827/train.parquet
Wrote: s3://incident-pipeline-test/gold/training/20260308_205827/valid.parquet
Wrote: s3://incident-pipeline-test/gold/training/20260308_205827/test.parquet
Wrote: s3://incident-pipeline-test/gold/training/20260308_205827/label_mapping.json


# 6) Build dataset card and final evidence


In [60]:
# Build a dataset card for feature/label/split/inference contract docs
rare_class_count = int(len(rare))
rare_mapped_count = int((df["label_final"] == MANUAL_REVIEW_GROUP).sum())
created_utc = datetime.now(timezone.utc).isoformat()

label_preview = sorted(id_to_label.items(), key=lambda x: x[0])[:20]
label_preview_lines = "\n".join([f"| {label_id} | {label_name} |" for label_id, label_name in label_preview])

dataset_card = f"""# Dataset card

## Overview
- dataset_version: {dataset_version}
- source_silver_key: s3://{SILVER_BUCKET}/{SILVER_OBJECT}
- created_utc: {created_utc}
- rows_total: {len(df)}
- split_sizes: train={len(train_df)}, valid={len(valid_df)}, test={len(test_df)}
- class_count: {len(classes)}
- rare_class_count: {rare_class_count}
- rare_mapped_count: {rare_mapped_count}

## Features (Feature dictionary)
- text = short_description + "\n" + description (fillna -> ""; strip)

## Label (Label dictionary)
- source field: assignment_group
- clean rule: strip whitespace
- rare class policy: labels with < {MIN_CLASS_COUNT} records -> {MANUAL_REVIEW_GROUP}
- full mapping lives in label_mapping.json

### Label preview (first 20)
| label_id | label_name |
|---:|---|
{label_preview_lines}

## Split rules
- 80/10/10 split on label_id
- random_state: {RANDOM_STATE}

## Inference input contract
- required: at least one of short_description or description
- preprocessing: text = short_description + "\n" + description (fillna -> ""; strip)
"""

dataset_card_bytes = dataset_card.encode("utf-8")
dataset_card_object = f"{GOLD_TRAINING_PREFIX}/dataset_card.md"
client.put_object(
    bucket_name=GOLD_BUCKET,
    object_name=dataset_card_object,
    data=io.BytesIO(dataset_card_bytes),
    length=len(dataset_card_bytes),
    content_type="text/markdown",
)
artifact_objects["dataset_card"] = dataset_card_object
print(f"Wrote: s3://{GOLD_BUCKET}/{dataset_card_object}")

# Final summary 
print("\nFinal evidence summary")
print("Rows total:", len(df))
print("Split sizes:", {"train": len(train_df), "valid": len(valid_df), "test": len(test_df)})
print("Class count:", len(classes))
print("Rare class count:", rare_class_count)
print("Rare mapped count:", rare_mapped_count)
print("Artifacts:")
for k, v in artifact_objects.items():
    print(f"- {k}: s3://{GOLD_BUCKET}/{v}")



Wrote: s3://incident-pipeline-test/gold/training/20260308_205827/dataset_card.md

Final evidence summary
Rows total: 10000
Split sizes: {'train': 7999, 'valid': 1001, 'test': 1000}
Class count: 12
Rare class count: 0
Rare mapped count: 0
Artifacts:
- train: s3://incident-pipeline-test/gold/training/20260308_205827/train.parquet
- valid: s3://incident-pipeline-test/gold/training/20260308_205827/valid.parquet
- test: s3://incident-pipeline-test/gold/training/20260308_205827/test.parquet
- label_mapping: s3://incident-pipeline-test/gold/training/20260308_205827/label_mapping.json
- dataset_card: s3://incident-pipeline-test/gold/training/20260308_205827/dataset_card.md
